# DrugBank XML Parsing

## Objective

The objective of this notebook is to extract useful information from the DrugBank XML database and convert it into a structured CSV file for machine learning.

---

## Workflow

Open XML

↓

Read one drug

↓

Extract required fields

↓

Write to CSV

↓

Repeat until the end

In [1]:
import xml.etree.ElementTree as ET
import csv

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_basic.csv"

NS = "{http://www.drugbank.ca}"

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)

    writer.writerow([
        "drugbank_id",
        "name",
        "description",
        "indication",
        "mechanism_of_action"
    ])

    for event, drug in ET.iterparse(XML_FILE, events=("end",)):
        if drug.tag == NS + "drug":

            drug_id = drug.findtext(NS + "drugbank-id", default="")
            name = drug.findtext(NS + "name", default="")
            description = drug.findtext(NS + "description", default="")
            indication = drug.findtext(NS + "indication", default="")
            mechanism = drug.findtext(NS + "mechanism-of-action", default="")

            writer.writerow([
                drug_id,
                name,
                description,
                indication,
                mechanism
            ])

            drug.clear()

print("Finished. CSV file created:")
print(OUTPUT_FILE)

Finished. CSV file created:
/Users/sankhadeep/Desktop/DrugBank/drugbank_basic.csv


# Step 1 - Reading DrugBank XML

## Objective

The objective of this notebook is to read the DrugBank XML database and extract only the DrugBank ID and Drug Name.

This is the first step in building a structured dataset for machine learning.

### Expected Output

A CSV file containing two columns:

- DrugBank ID
- Drug Name

In [2]:
import xml.etree.ElementTree as ET
import csv

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"

OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step1.csv"

NS = "{http://www.drugbank.ca}"

csvfile = open(OUTPUT_FILE, "w", newline="", encoding="utf-8")

writer = csv.writer(csvfile)

writer.writerow(["DrugBank_ID", "Drug_Name"])

print("CSV created successfully.")


context = ET.iterparse(XML_FILE, events=("end",))

for event, elem in context:

    if elem.tag == NS + "drug":

        drug_id = elem.findtext(NS + "drugbank-id")

        drug_name = elem.findtext(NS + "name")

        print(drug_id, drug_name)

        break


writer.writerow([drug_id, drug_name])

csvfile.close()

print("Finished.")

CSV created successfully.
DB00001 Lepirudin
Finished.


# Step 2 – Extract All DrugBank IDs and Drug Names

## Objective

The objective of this step is to read the complete DrugBank XML database and extract the DrugBank ID and Drug Name for every drug. Unlike Step 1, where only the first drug record was extracted for testing purposes, this step processes the entire XML file.

The extracted information is stored in a CSV file, which serves as the first structured dataset for subsequent preprocessing and machine learning tasks.

### Workflow

1. Import the required Python libraries.
2. Specify the location of the DrugBank XML file.
3. Create a CSV file with appropriate column headers.
4. Parse the XML file sequentially using `iterparse()`.
5. For each `<drug>` element:
   - Extract the DrugBank ID.
   - Extract the Drug Name.
   - Write the extracted values into the CSV file.
6. Count the total number of drugs extracted.
7. Save the CSV file.
8. Verify the output by displaying the first five records, the dataset dimensions, and the last five records.

### Expected Output

A CSV file named **drugbank_step2_all_drugs.csv** containing two columns:

- DrugBank_ID
- Drug_Name

The number of extracted rows should approximately equal the total number of drug records present in the DrugBank XML database.

In [3]:
# ============================================================
# Step 2 - Extract All DrugBank IDs and Drug Names
# ============================================================

# Import required libraries
import xml.etree.ElementTree as ET
import csv
import pandas as pd

# ------------------------------------------------------------
# Define file paths
# ------------------------------------------------------------

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"

OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step2_all_drugs.csv"

# DrugBank XML Namespace
NS = "{http://www.drugbank.ca}"

# Counter for total drugs
count = 0

print("Creating CSV file...")

# ------------------------------------------------------------
# Create CSV file
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    # CSV Header
    writer.writerow([
        "DrugBank_ID",
        "Drug_Name"
    ])

    print("Reading DrugBank XML...")

    # --------------------------------------------------------
    # Read the XML sequentially
    # --------------------------------------------------------

    for event, elem in ET.iterparse(XML_FILE, events=("end",)):

        # Process only <drug> elements
        if elem.tag == NS + "drug":

            # Extract DrugBank ID
            drug_id = elem.findtext(NS + "drugbank-id", default="")

            # Extract Drug Name
            drug_name = elem.findtext(NS + "name", default="")

            # Write into CSV
            writer.writerow([
                drug_id,
                drug_name
            ])

            # Increase counter
            count += 1

            # Free memory
            elem.clear()

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)
print("CSV saved at:")
print(OUTPUT_FILE)

# ------------------------------------------------------------
# Verify the extracted CSV
# ------------------------------------------------------------

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 3963134
CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step2_all_drugs.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID      Drug_Name
0     DB00001      Lepirudin
1     DB01022  Phylloquinone
2     DB01373        Calcium
3     DB00001      Lepirudin
4     DB00002      Cetuximab

Dataset Shape
(3963134, 2)

Last Five Records
        DrugBank_ID                     Drug_Name
3963129     DB32724                       SCY-247
3963130     DB32725              Tripentadecanoin
3963131     DB32726  Propylene glycol dicaprylate
3963132     DB32727                      RCT-1213
3963133     DB32728                 Cerous cation


# XML Structure Inspection

## Objective

Before extracting data from the DrugBank XML database, it is important to understand its hierarchical structure.

The purpose of this notebook is to inspect the XML tree and determine where the top-level `<drug>` elements are located.

Understanding the XML hierarchy helps avoid extracting nested elements unintentionally.

In [4]:
import xml.etree.ElementTree as ET

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"

tree = ET.parse(XML_FILE)

root = tree.getroot()

print("Root tag:")
print(root.tag)

print()

print("First 20 child elements of the root:\n")

for child in list(root)[:20]:
    print(child.tag)

Root tag:
{http://www.drugbank.ca}drugbank

First 20 child elements of the root:

{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug
{http://www.drugbank.ca}drug


# Step 2 (Corrected) – Extract Only the Top-Level Drug Records

## Objective

In the previous step, every XML element named `<drug>` was extracted, resulting in approximately 3.9 million records. However, DrugBank contains only around 17,000 unique drugs. This indicates that the XML file contains additional nested `<drug>` elements that should not be treated as independent drug records.

The objective of this step is to extract only the **top-level `<drug>` elements**, which are direct children of the root `<drugbank>` element.

To achieve this, the XML tree depth is monitored while parsing the document. Only `<drug>` elements that occur at the second level of the XML hierarchy are processed.

The following operations are performed:

1. Import the required Python libraries.
2. Define the locations of the XML and CSV files.
3. Create an output CSV file.
4. Read the DrugBank XML sequentially using `iterparse()`.
5. Track the XML depth during parsing.
6. Extract only top-level drug records.
7. Store the DrugBank ID and Drug Name in a CSV file.
8. Count the number of extracted drugs.
9. Load the generated CSV using pandas for verification.

## Expected Output

A CSV file named **drugbank_step2_corrected.csv** containing only the unique top-level DrugBank records.

The output should contain approximately **17,000** rows with two columns:

- DrugBank_ID
- Drug_Name

This dataset will serve as the foundation for extracting additional attributes such as descriptions, indications, targets, pathways, enzymes, transporters and drug interactions in the subsequent steps.

In [5]:
# ============================================================
# Step 2 (Corrected)
# Extract Only Top-Level DrugBank Drug Records
# ============================================================

# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

import xml.etree.ElementTree as ET
import csv
import pandas as pd

# ------------------------------------------------------------
# Define File Locations
# ------------------------------------------------------------

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"

OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step2_corrected.csv"

# DrugBank XML Namespace
NS = "{http://www.drugbank.ca}"

# ------------------------------------------------------------
# Initialize Counter
# ------------------------------------------------------------

count = 0

print("Creating CSV file...")

# ------------------------------------------------------------
# Create CSV File
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    # Write CSV Header
    writer.writerow([
        "DrugBank_ID",
        "Drug_Name"
    ])

    print("Reading DrugBank XML...")

    # --------------------------------------------------------
    # Parse XML
    # --------------------------------------------------------

    context = ET.iterparse(
        XML_FILE,
        events=("start", "end")
    )

    depth = 0

    for event, elem in context:

        # Entering an XML element
        if event == "start":
            depth += 1

        # Leaving an XML element
        elif event == "end":

            # Process only top-level <drug> elements
            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(
                    NS + "drugbank-id",
                    default=""
                )

                drug_name = elem.findtext(
                    NS + "name",
                    default=""
                )

                writer.writerow([
                    drug_id,
                    drug_name
                ])

                count += 1

                # Release memory
                elem.clear()

            depth -= 1

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

# ------------------------------------------------------------
# Verify the Output
# ------------------------------------------------------------

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step2_corrected.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name
0     DB00001            Lepirudin
1     DB00002            Cetuximab
2     DB00003         Dornase alfa
3     DB00004  Denileukin diftitox
4     DB00005           Etanercept

Dataset Shape
(19856, 2)

Last Five Records
      DrugBank_ID                     Drug_Name
19851     DB32724                       SCY-247
19852     DB32725              Tripentadecanoin
19853     DB32726  Propylene glycol dicaprylate
19854     DB32727                      RCT-1213
19855     DB32728                 Cerous cation


# Step 3 – Extract DrugBank ID, Drug Name, and Description

## Objective

The objective of this step is to extend the corrected DrugBank XML parser by extracting the drug description along with the DrugBank ID and Drug Name.

The drug description provides useful background information about each drug, including its biological, chemical, or therapeutic characteristics. This information may later support feature engineering, text analysis, or natural language processing tasks in the drug repurposing workflow.

This step performs the following operations:

1. Read the DrugBank XML file.
2. Process only top-level `<drug>` records.
3. Extract:
   - DrugBank ID
   - Drug Name
   - Description
4. Save the extracted data into a CSV file.
5. Verify the output using pandas.

## Expected Output

A CSV file named **drugbank_step3_description.csv** containing three columns:

- DrugBank_ID
- Drug_Name
- Description

The number of rows should remain the same as Step 2, approximately **19,856** records.

In [6]:
# ============================================================
# Step 3
# Extract DrugBank ID, Drug Name, and Description
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step3_description.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")

                writer.writerow([
                    drug_id,
                    drug_name,
                    description
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...



Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step3_description.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  
0  Lepirudin is a recombinant hirudin formed by 6...  
1  Cetuximab is a recombinant chimeric human/mous...  
2  Dornase alfa is a biosynthetic form of human d...  
3  Denileukin diftitox is an IL2-receptor-directe...  
4  Dimeric fusion protein consisting of the extra...  

Dataset Shape
(19856, 3)

Missing Values
DrugBank_ID       0
Drug_Name         0
Description    7406
dtype: int64

Last Five Records
      DrugBank_ID                     Drug_Name  \
19851     DB32724       

# Step 4 – Extract DrugBank ID, Drug Name, Description, and Indication

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting the **Indication** field in addition to the DrugBank ID, Drug Name, and Description.

The indication field is especially important for drug repurposing because it describes the current therapeutic use of a drug, including the diseases or medical conditions for which the drug is used.

This step performs the following operations:

1. Read the DrugBank XML file.
2. Process only top-level `<drug>` records.
3. Extract:
   - DrugBank ID
   - Drug Name
   - Description
   - Indication
4. Save the extracted data into a CSV file.
5. Verify the output using pandas.

## Expected Output

A CSV file named **drugbank_step4_indication.csv** containing four columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication

The number of rows should remain the same as the previous step, approximately **19,856** records.

In [7]:
# ============================================================
# Step 4
# Extract DrugBank ID, Drug Name, Description, and Indication
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step4_indication.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step4_indication.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  
0  Lepirudin is indicated for anticoagulation in ...  
1  Cetuximab is indicated for: \r\n\r\n- The tre

# Step 5 – Extract DrugBank ID, Drug Name, Description, Indication, and Mechanism of Action

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting the **Mechanism of Action** field in addition to the DrugBank ID, Drug Name, Description, and Indication.

The mechanism of action explains how a drug produces its biological or therapeutic effect. This information is important for drug repurposing because drugs with similar mechanisms may be useful for treating similar or related diseases.

This step performs the following operations:

1. Read the DrugBank XML file.
2. Process only top-level `<drug>` records.
3. Extract:
   - DrugBank ID
   - Drug Name
   - Description
   - Indication
   - Mechanism of Action
4. Save the extracted data into a CSV file.
5. Verify the output using pandas.

## Expected Output

A CSV file named **drugbank_step5_mechanism.csv** containing five columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action

The number of rows should remain the same as the previous step, approximately **19,856** records.

In [8]:
# ============================================================
# Step 5
# Extract DrugBank ID, Drug Name, Description, Indication,
# and Mechanism of Action
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step5_mechanism.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step5_mechanism.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The tr

# Step 6 – Extract Drug Targets

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting **drug target information**.

Drug targets are important for drug repurposing because they describe the proteins or biological molecules that a drug acts on. Drugs that share similar targets may have similar therapeutic effects or may be candidates for new disease applications.

This step extracts:

- DrugBank ID
- Drug Name
- Description
- Indication
- Mechanism of Action
- Target Names

Because one drug may have multiple targets, all target names for a drug are combined into a single text field separated by semicolons.

## Expected Output

A CSV file named **drugbank_step6_targets.csv** containing six columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action
- Targets

The number of rows should remain approximately **19,856** records.

In [9]:
# ============================================================
# Step 6
# Extract DrugBank ID, Drug Name, Description, Indication,
# Mechanism of Action, and Drug Targets
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step6_targets.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action",
        "Targets"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                target_names = []

                targets_element = elem.find(NS + "targets")

                if targets_element is not None:
                    for target in targets_element.findall(NS + "target"):
                        target_name = target.findtext(NS + "name", default="")
                        if target_name:
                            target_names.append(target_name)

                targets_text = "; ".join(target_names)

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism,
                    targets_text
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step6_targets.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The trea

# Step 7 – Extract Drug Pathways

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting **pathway information**.

Drug pathways are important for drug repurposing because they describe the biological or metabolic pathways associated with a drug. Pathway information can help identify relationships between drugs, biological processes, and disease mechanisms.

This step extracts:

- DrugBank ID
- Drug Name
- Description
- Indication
- Mechanism of Action
- Targets
- Pathways

Because one drug may be associated with multiple pathways, all pathway names for each drug are combined into a single text field separated by semicolons.

## Expected Output

A CSV file named **drugbank_step7_pathways.csv** containing seven columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action
- Targets
- Pathways

The number of rows should remain approximately **19,856** records.

In [10]:
# ============================================================
# Step 7
# Extract DrugBank ID, Drug Name, Description, Indication,
# Mechanism of Action, Targets, and Pathways
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step7_pathways.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action",
        "Targets",
        "Pathways"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                target_names = []
                targets_element = elem.find(NS + "targets")

                if targets_element is not None:
                    for target in targets_element.findall(NS + "target"):
                        target_name = target.findtext(NS + "name", default="")
                        if target_name:
                            target_names.append(target_name)

                targets_text = "; ".join(target_names)

                pathway_names = []
                pathways_element = elem.find(NS + "pathways")

                if pathways_element is not None:
                    for pathway in pathways_element.findall(NS + "pathway"):
                        pathway_name = pathway.findtext(NS + "name", default="")
                        if pathway_name:
                            pathway_names.append(pathway_name)

                pathways_text = "; ".join(pathway_names)

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism,
                    targets_text,
                    pathways_text
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step7_pathways.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The tre

# Step 8 – Extract Drug Categories

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting **drug category information**.

Drug categories are useful for drug repurposing because they describe the pharmacological or therapeutic class of a drug. Drugs belonging to similar categories may share related biological effects or therapeutic applications.

This step extracts:

- DrugBank ID
- Drug Name
- Description
- Indication
- Mechanism of Action
- Targets
- Pathways
- Categories

Because one drug may belong to multiple categories, all category names for each drug are combined into a single text field separated by semicolons.

## Expected Output

A CSV file named **drugbank_step8_categories.csv** containing eight columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action
- Targets
- Pathways
- Categories

The number of rows should remain approximately **19,856** records.

In [11]:
# ============================================================
# Step 8
# Extract DrugBank ID, Drug Name, Description, Indication,
# Mechanism of Action, Targets, Pathways, and Categories
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step8_categories.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action",
        "Targets",
        "Pathways",
        "Categories"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                target_names = []
                targets_element = elem.find(NS + "targets")

                if targets_element is not None:
                    for target in targets_element.findall(NS + "target"):
                        target_name = target.findtext(NS + "name", default="")
                        if target_name:
                            target_names.append(target_name)

                targets_text = "; ".join(target_names)

                pathway_names = []
                pathways_element = elem.find(NS + "pathways")

                if pathways_element is not None:
                    for pathway in pathways_element.findall(NS + "pathway"):
                        pathway_name = pathway.findtext(NS + "name", default="")
                        if pathway_name:
                            pathway_names.append(pathway_name)

                pathways_text = "; ".join(pathway_names)

                category_names = []
                categories_element = elem.find(NS + "categories")

                if categories_element is not None:
                    for category in categories_element.findall(NS + "category"):
                        category_name = category.findtext(NS + "category", default="")
                        if category_name:
                            category_names.append(category_name)

                categories_text = "; ".join(category_names)

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism,
                    targets_text,
                    pathways_text,
                    categories_text
                ])

                count += 1
                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step8_categories.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The t

# Step 9 – Extract ATC Codes

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting the Anatomical Therapeutic Chemical (ATC) classification codes.

The ATC classification system is an internationally recognized drug classification system developed by the World Health Organization (WHO). It categorizes drugs according to the organ or system on which they act, their therapeutic use, pharmacological properties, and chemical characteristics.

This information is valuable for drug repurposing because drugs belonging to similar ATC classes often exhibit similar therapeutic effects and mechanisms of action.

This step extracts:

- DrugBank ID
- Drug Name
- Description
- Indication
- Mechanism of Action
- Targets
- Pathways
- Categories
- ATC Codes

Since a drug may have multiple ATC codes, all codes are combined into a single text field separated by semicolons.

## Expected Output

A CSV file named **drugbank_step9_atc.csv** containing the following columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action
- Targets
- Pathways
- Categories
- ATC_Codes

The number of rows should remain approximately **19,856**.

In [12]:
# ============================================================
# Step 9
# Extract DrugBank ID, Drug Name, Description, Indication,
# Mechanism of Action, Targets, Pathways, Categories,
# and ATC Codes
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step9_atc.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action",
        "Targets",
        "Pathways",
        "Categories",
        "ATC_Codes"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                # ---------------- Targets ----------------

                target_names = []

                targets_element = elem.find(NS + "targets")

                if targets_element is not None:
                    for target in targets_element.findall(NS + "target"):
                        target_name = target.findtext(NS + "name", default="")
                        if target_name:
                            target_names.append(target_name)

                targets_text = "; ".join(target_names)

                # ---------------- Pathways ----------------

                pathway_names = []

                pathways_element = elem.find(NS + "pathways")

                if pathways_element is not None:
                    for pathway in pathways_element.findall(NS + "pathway"):
                        pathway_name = pathway.findtext(NS + "name", default="")
                        if pathway_name:
                            pathway_names.append(pathway_name)

                pathways_text = "; ".join(pathway_names)

                # ---------------- Categories ----------------

                category_names = []

                categories_element = elem.find(NS + "categories")

                if categories_element is not None:
                    for category in categories_element.findall(NS + "category"):
                        category_name = category.findtext(NS + "category", default="")
                        if category_name:
                            category_names.append(category_name)

                categories_text = "; ".join(category_names)

                # ---------------- ATC Codes ----------------

                atc_codes = []

                atc_element = elem.find(NS + "atc-codes")

                if atc_element is not None:
                    for atc in atc_element.findall(NS + "atc-code"):

                        code = atc.get("code")

                        if code:
                            atc_codes.append(code)

                atc_text = "; ".join(atc_codes)

                # ---------------- Write Row ----------------

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism,
                    targets_text,
                    pathways_text,
                    categories_text,
                    atc_text
                ])

                count += 1

                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step9_atc.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The trea... 

# Step 10 – Exploratory Data Assessment of the Extracted DrugBank Dataset

## Objective

The objective of this step is to examine the structured DrugBank dataset created in the previous steps.

Before using the dataset for machine learning, it is important to check:

1. Dataset size
2. Column names
3. Missing values
4. Duplicate DrugBank IDs
5. Number of available values in each feature
6. Basic quality of extracted fields

This step helps verify whether the extracted dataset is suitable for further preprocessing and feature engineering.

In [13]:
# ============================================================
# Step 10
# Exploratory Data Assessment of DrugBank Extracted Dataset
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Load the extracted dataset
# ------------------------------------------------------------

FILE_PATH = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step9_atc.csv"

df = pd.read_csv(FILE_PATH)

# ------------------------------------------------------------
# 1. Dataset shape
# ------------------------------------------------------------

print("Dataset Shape:")
print(df.shape)

# ------------------------------------------------------------
# 2. Column names
# ------------------------------------------------------------

print("\nColumn Names:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 3. First five records
# ------------------------------------------------------------

print("\nFirst Five Records:")
print(df.head())

# ------------------------------------------------------------
# 4. Last five records
# ------------------------------------------------------------

print("\nLast Five Records:")
print(df.tail())

# ------------------------------------------------------------
# 5. Missing values
# ------------------------------------------------------------

print("\nMissing Values:")
print(df.isnull().sum())

# ------------------------------------------------------------
# 6. Missing value percentage
# ------------------------------------------------------------

print("\nMissing Value Percentage:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage.round(2))

# ------------------------------------------------------------
# 7. Duplicate DrugBank IDs
# ------------------------------------------------------------

duplicate_ids = df["DrugBank_ID"].duplicated().sum()

print("\nDuplicate DrugBank IDs:")
print(duplicate_ids)

# ------------------------------------------------------------
# 8. Number of unique drugs
# ------------------------------------------------------------

unique_drugs = df["DrugBank_ID"].nunique()

print("\nNumber of Unique DrugBank IDs:")
print(unique_drugs)

# ------------------------------------------------------------
# 9. Count available values per column
# ------------------------------------------------------------

print("\nNon-Missing Values Per Column:")
print(df.notnull().sum())

# ------------------------------------------------------------
# 10. Summary
# ------------------------------------------------------------

print("\nSummary:")
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Unique drugs:", unique_drugs)
print("Duplicate IDs:", duplicate_ids)

Dataset Shape:
(19856, 9)

Column Names:
['DrugBank_ID', 'Drug_Name', 'Description', 'Indication', 'Mechanism_of_Action', 'Targets', 'Pathways', 'Categories', 'ATC_Codes']

First Five Records:
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The trea...   
2  Used as adjunct therapy in the treatment of cy...   
3  Denileukin dif

# Understanding SMILES

## What is SMILES?

SMILES (Simplified Molecular Input Line Entry System) is a text-based notation used to represent the chemical structure of a molecule. Instead of storing a molecule as a graphical chemical diagram, SMILES encodes the molecular structure as a sequence of characters that can be easily processed by computers.

For example:

| Molecule | SMILES |
|----------|---------|
| Water | `O` |
| Ethanol | `CCO` |
| Benzene | `c1ccccc1` |
| Aspirin | `CC(=O)OC1=CC=CC=C1C(=O)O` |

Each character or symbol represents atoms, chemical bonds, branching, ring structures, and other molecular properties.

## Why is SMILES Important?

SMILES is one of the most widely used molecular representations in cheminformatics and computational drug discovery. Since machine learning algorithms cannot directly process chemical structure diagrams, SMILES provides a standardized text representation that can be used for computational analysis.

Using SMILES, researchers can calculate a wide range of molecular descriptors, including:

- Molecular weight
- Number of atoms
- Number of rings
- Hydrogen bond donors
- Hydrogen bond acceptors
- LogP (lipophilicity)
- Molecular fingerprints
- Topological descriptors

These descriptors are commonly used as input features for machine learning models in drug discovery and drug repurposing.

## Why is SMILES Relevant to This Dissertation?

The objective of this dissertation is to identify potential opportunities for drug repurposing using machine learning techniques. Although the current study focuses on extracting information from DrugBank, the SMILES representation provides an important link between the biological information stored in DrugBank and the chemical structure of each drug.

By extracting the SMILES representation, it becomes possible to generate molecular descriptors and fingerprints using cheminformatics libraries such as RDKit. These features can later be incorporated into machine learning models to improve prediction performance and enable structure-based analysis of drug similarity.

## Where is SMILES Stored in DrugBank?

Unlike fields such as Drug Name or Indication, the SMILES representation is not stored as a direct child element of each drug. Instead, it is stored within the **Calculated Properties** section of the DrugBank XML file.

The XML structure is similar to the following:

```xml
<calculated-properties>
    <property>
        <kind>SMILES</kind>
        <value>CC(=O)OC1=CC=CC=C1C(=O)O</value>
    </property>
</calculated-properties>
```

Therefore, extracting SMILES requires searching through the calculated properties of each drug and identifying the property whose `<kind>` element is equal to **SMILES**.

# Step 11 – Extract SMILES Representation

## Objective

The objective of this step is to extend the DrugBank XML parser by extracting the **SMILES (Simplified Molecular Input Line Entry System)** representation for each drug.

SMILES is a text-based representation of a molecule's chemical structure and is one of the most widely used molecular representations in cheminformatics. Unlike the descriptive information extracted in the previous steps, SMILES provides structural information about each drug molecule.

The extracted SMILES strings can later be used to generate molecular descriptors, molecular fingerprints, and other chemical features using cheminformatics libraries such as RDKit. These features are commonly used as inputs to machine learning models for drug discovery and drug repurposing.

This step performs the following operations:

1. Read the DrugBank XML file.
2. Process only the top-level `<drug>` elements.
3. Extract:
   - DrugBank ID
   - Drug Name
   - Description
   - Indication
   - Mechanism of Action
   - Targets
   - Pathways
   - Categories
   - ATC Codes
   - SMILES
4. Save the extracted information into a CSV file.
5. Verify the extracted dataset using pandas.

## Expected Output

A CSV file named **drugbank_step11_smiles.csv** containing ten columns:

- DrugBank_ID
- Drug_Name
- Description
- Indication
- Mechanism_of_Action
- Targets
- Pathways
- Categories
- ATC_Codes
- SMILES

The number of rows should remain approximately **19,856** records.

In [14]:
# ============================================================
# Step 11
# Extract DrugBank ID, Drug Name, Description, Indication,
# Mechanism of Action, Targets, Pathways, Categories,
# ATC Codes and SMILES
# ============================================================

import xml.etree.ElementTree as ET
import csv
import pandas as pd

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"
OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step11_smiles.csv"

NS = "{http://www.drugbank.ca}"

count = 0

print("Creating CSV file...")

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:

    writer = csv.writer(csvfile)

    writer.writerow([
        "DrugBank_ID",
        "Drug_Name",
        "Description",
        "Indication",
        "Mechanism_of_Action",
        "Targets",
        "Pathways",
        "Categories",
        "ATC_Codes",
        "SMILES"
    ])

    print("Reading DrugBank XML...")

    context = ET.iterparse(XML_FILE, events=("start", "end"))

    depth = 0

    for event, elem in context:

        if event == "start":
            depth += 1

        elif event == "end":

            if elem.tag == NS + "drug" and depth == 2:

                drug_id = elem.findtext(NS + "drugbank-id", default="")
                drug_name = elem.findtext(NS + "name", default="")
                description = elem.findtext(NS + "description", default="")
                indication = elem.findtext(NS + "indication", default="")
                mechanism = elem.findtext(NS + "mechanism-of-action", default="")

                # ---------------- Targets ----------------

                target_names = []

                targets_element = elem.find(NS + "targets")

                if targets_element is not None:
                    for target in targets_element.findall(NS + "target"):
                        target_name = target.findtext(NS + "name", default="")
                        if target_name:
                            target_names.append(target_name)

                targets_text = "; ".join(target_names)

                # ---------------- Pathways ----------------

                pathway_names = []

                pathways_element = elem.find(NS + "pathways")

                if pathways_element is not None:
                    for pathway in pathways_element.findall(NS + "pathway"):
                        pathway_name = pathway.findtext(NS + "name", default="")
                        if pathway_name:
                            pathway_names.append(pathway_name)

                pathways_text = "; ".join(pathway_names)

                # ---------------- Categories ----------------

                category_names = []

                categories_element = elem.find(NS + "categories")

                if categories_element is not None:
                    for category in categories_element.findall(NS + "category"):
                        category_name = category.findtext(NS + "category", default="")
                        if category_name:
                            category_names.append(category_name)

                categories_text = "; ".join(category_names)

                # ---------------- ATC Codes ----------------

                atc_codes = []

                atc_element = elem.find(NS + "atc-codes")

                if atc_element is not None:
                    for atc in atc_element.findall(NS + "atc-code"):
                        code = atc.get("code")
                        if code:
                            atc_codes.append(code)

                atc_text = "; ".join(atc_codes)

                # ---------------- SMILES ----------------

                smiles = ""

                calculated_properties = elem.find(NS + "calculated-properties")

                if calculated_properties is not None:

                    for prop in calculated_properties.findall(NS + "property"):

                        kind = prop.findtext(NS + "kind", default="")

                        if kind == "SMILES":

                            smiles = prop.findtext(NS + "value", default="")

                            break

                # ---------------- Write Row ----------------

                writer.writerow([
                    drug_id,
                    drug_name,
                    description,
                    indication,
                    mechanism,
                    targets_text,
                    pathways_text,
                    categories_text,
                    atc_text,
                    smiles
                ])

                count += 1

                elem.clear()

            depth -= 1

print("\nExtraction Completed Successfully")
print("--------------------------------")
print("Total Drugs Extracted :", count)

print("\nCSV saved at:")
print(OUTPUT_FILE)

print("\nLoading CSV for verification...")

df = pd.read_csv(OUTPUT_FILE)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nLast Five Records")
print(df.tail())

Creating CSV file...
Reading DrugBank XML...

Extraction Completed Successfully
--------------------------------
Total Drugs Extracted : 19856

CSV saved at:
/Users/sankhadeep/Desktop/DrugBank/drugbank_step11_smiles.csv

Loading CSV for verification...

First Five Records
  DrugBank_ID            Drug_Name  \
0     DB00001            Lepirudin   
1     DB00002            Cetuximab   
2     DB00003         Dornase alfa   
3     DB00004  Denileukin diftitox   
4     DB00005           Etanercept   

                                         Description  \
0  Lepirudin is a recombinant hirudin formed by 6...   
1  Cetuximab is a recombinant chimeric human/mous...   
2  Dornase alfa is a biosynthetic form of human d...   
3  Denileukin diftitox is an IL2-receptor-directe...   
4  Dimeric fusion protein consisting of the extra...   

                                          Indication  \
0  Lepirudin is indicated for anticoagulation in ...   
1  Cetuximab is indicated for: \r\n\r\n- The trea

# Step 12 – Data Cleaning and Preparation

## Objective

The objective of this step is to clean and prepare the DrugBank dataset extracted in Step 11 before proceeding to feature engineering and machine learning.

The previous data assessment confirmed that the dataset contains 19,856 unique DrugBank records with no duplicate DrugBank IDs. Therefore, duplicate DrugBank IDs are checked again in this step for verification.

Several extracted attributes contain missing values. Instead of removing drugs with missing information at this stage, the missing text values are standardized and additional indicator columns are created to identify the availability of important information for each drug.

This step performs the following operations:

1. Load the DrugBank dataset generated in Step 11.
2. Verify the number of duplicate and unique DrugBank IDs.
3. Standardize missing and blank values in the extracted text fields.
4. Create indicator columns to identify the availability of:
   - SMILES representations
   - Therapeutic indications
   - Drug targets
   - Pathways
   - Drug categories
   - ATC codes
   - Mechanism of action
   - Drug descriptions
5. Calculate the number and percentage of drugs for which each type of information is available.
6. Generate a summary table describing data availability.
7. Save the cleaned dataset for subsequent analysis and feature engineering.

## Expected Output

A cleaned DrugBank dataset named **drugbank_step12_cleaned.csv**.

The original DrugBank attributes are retained, and the following data availability indicator columns are added:

- Has_SMILES
- Has_Indication
- Has_Targets
- Has_Pathways
- Has_Categories
- Has_ATC
- Has_Mechanism
- Has_Description

No drug records are removed solely because of missing information at this stage. The cleaned dataset will be used in the subsequent stages to determine suitable records and features for drug repurposing analysis.

In [15]:
# ============================================================
# Step 12 – Data Cleaning and Preparation
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Define file paths
# ------------------------------------------------------------

INPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step11_smiles.csv"

OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step12_cleaned.csv"


# ------------------------------------------------------------
# Load the dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("Original Dataset Shape:")
print(df.shape)


# ------------------------------------------------------------
# Check duplicate DrugBank IDs
# ------------------------------------------------------------

duplicate_count = df["DrugBank_ID"].duplicated().sum()

print("\nDuplicate DrugBank IDs:")
print(duplicate_count)

if duplicate_count == 0:
    print("No duplicate DrugBank IDs found.")
else:
    print("Warning: Duplicate DrugBank IDs are present.")


# ------------------------------------------------------------
# Check number of unique DrugBank IDs
# ------------------------------------------------------------

unique_drugs = df["DrugBank_ID"].nunique()

print("\nNumber of Unique DrugBank IDs:")
print(unique_drugs)


# ------------------------------------------------------------
# Standardise missing and blank text values
# ------------------------------------------------------------

text_columns = [
    "Drug_Name",
    "Description",
    "Indication",
    "Mechanism_of_Action",
    "Targets",
    "Pathways",
    "Categories",
    "ATC_Codes",
    "SMILES"
]

for column in text_columns:

    # Replace missing values with empty strings
    df[column] = df[column].fillna("")

    # Convert values to strings and remove
    # leading/trailing whitespace
    df[column] = df[column].astype(str).str.strip()


# ------------------------------------------------------------
# Create data availability indicators
# ------------------------------------------------------------

df["Has_SMILES"] = df["SMILES"].ne("")

df["Has_Indication"] = df["Indication"].ne("")

df["Has_Targets"] = df["Targets"].ne("")

df["Has_Pathways"] = df["Pathways"].ne("")

df["Has_Categories"] = df["Categories"].ne("")

df["Has_ATC"] = df["ATC_Codes"].ne("")

df["Has_Mechanism"] = df["Mechanism_of_Action"].ne("")

df["Has_Description"] = df["Description"].ne("")


# ------------------------------------------------------------
# Display data availability statistics
# ------------------------------------------------------------

print("\nData Availability")
print("-----------------------------")

print("Total Drugs:",
      len(df))

print("Drugs with SMILES:",
      df["Has_SMILES"].sum())

print("Drugs with Indication:",
      df["Has_Indication"].sum())

print("Drugs with Targets:",
      df["Has_Targets"].sum())

print("Drugs with Pathways:",
      df["Has_Pathways"].sum())

print("Drugs with Categories:",
      df["Has_Categories"].sum())

print("Drugs with ATC Codes:",
      df["Has_ATC"].sum())

print("Drugs with Mechanism of Action:",
      df["Has_Mechanism"].sum())

print("Drugs with Description:",
      df["Has_Description"].sum())


# ------------------------------------------------------------
# Calculate availability percentages
# ------------------------------------------------------------

print("\nData Availability Percentage")
print("-----------------------------")

availability_columns = [
    "Has_SMILES",
    "Has_Indication",
    "Has_Targets",
    "Has_Pathways",
    "Has_Categories",
    "Has_ATC",
    "Has_Mechanism",
    "Has_Description"
]

for column in availability_columns:

    percentage = df[column].mean() * 100

    print(
        f"{column}: {percentage:.2f}%"
    )


# ------------------------------------------------------------
# Create data availability summary table
# ------------------------------------------------------------

summary = pd.DataFrame({

    "Feature": [
        "SMILES",
        "Indication",
        "Targets",
        "Pathways",
        "Categories",
        "ATC Codes",
        "Mechanism of Action",
        "Description"
    ],

    "Available_Count": [
        df["Has_SMILES"].sum(),
        df["Has_Indication"].sum(),
        df["Has_Targets"].sum(),
        df["Has_Pathways"].sum(),
        df["Has_Categories"].sum(),
        df["Has_ATC"].sum(),
        df["Has_Mechanism"].sum(),
        df["Has_Description"].sum()
    ]
})


# ------------------------------------------------------------
# Calculate availability percentage
# ------------------------------------------------------------

summary["Available_Percentage"] = (
    summary["Available_Count"] / len(df)
) * 100

summary["Available_Percentage"] = (
    summary["Available_Percentage"].round(2)
)


print("\nData Availability Summary:")
print(summary)


# ------------------------------------------------------------
# Check final dataset shape
# ------------------------------------------------------------

print("\nCleaned Dataset Shape:")
print(df.shape)


# ------------------------------------------------------------
# Preview cleaned dataset
# ------------------------------------------------------------

print("\nFirst Five Records:")
print(df.head())


# ------------------------------------------------------------
# Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nCleaned dataset saved at:")
print(OUTPUT_FILE)

Original Dataset Shape:
(19856, 10)

Duplicate DrugBank IDs:
0
No duplicate DrugBank IDs found.

Number of Unique DrugBank IDs:
19856

Data Availability
-----------------------------
Total Drugs: 19856
Drugs with SMILES: 14626
Drugs with Indication: 4484
Drugs with Targets: 10030
Drugs with Pathways: 1788
Drugs with Categories: 11140
Drugs with ATC Codes: 3636
Drugs with Mechanism of Action: 4257
Drugs with Description: 12450

Data Availability Percentage
-----------------------------
Has_SMILES: 73.66%
Has_Indication: 22.58%
Has_Targets: 50.51%
Has_Pathways: 9.00%
Has_Categories: 56.10%
Has_ATC: 18.31%
Has_Mechanism: 21.44%
Has_Description: 62.70%

Data Availability Summary:
               Feature  Available_Count  Available_Percentage
0               SMILES            14626                 73.66
1           Indication             4484                 22.58
2              Targets            10030                 50.51
3             Pathways             1788                  9.00
4    

In [16]:
# Check NaN values after cleaning
print("NaN values after cleaning:")
print(df[text_columns].isna().sum())

# Check empty strings after cleaning
print("\nEmpty strings after cleaning:")
print(df[text_columns].eq("").sum())

NaN values after cleaning:
Drug_Name              0
Description            0
Indication             0
Mechanism_of_Action    0
Targets                0
Pathways               0
Categories             0
ATC_Codes              0
SMILES                 0
dtype: int64

Empty strings after cleaning:
Drug_Name                  0
Description             7406
Indication             15372
Mechanism_of_Action    15599
Targets                 9826
Pathways               18068
Categories              8716
ATC_Codes              16220
SMILES                  5230
dtype: int64


# Step 13 – SMILES Validation

## Objective

The objective of this step is to validate the SMILES representations extracted from DrugBank before using them for molecular feature generation.

Step 12 showed that SMILES information is available for a substantial proportion of the DrugBank records. However, the presence of a SMILES value does not necessarily guarantee that the molecular representation is valid and can be successfully processed by cheminformatics tools.

In this step, the **RDKit** library is used to parse each available SMILES string and determine whether it represents a valid molecular structure.

Each DrugBank record is classified into one of the following categories:

- **Valid** – a SMILES value is available and can be successfully parsed by RDKit.
- **Invalid** – a SMILES value is available but cannot be parsed by RDKit.
- **Missing** – no SMILES value is available for the drug.

This step performs the following operations:

1. Load the cleaned DrugBank dataset generated in Step 12.
2. Check whether a SMILES representation is available for each drug.
3. Use RDKit to validate each available SMILES string.
4. Create a `Valid_SMILES` indicator column.
5. Create a `SMILES_Status` column to distinguish between valid, invalid, and missing SMILES.
6. Calculate the number and percentage of drugs in each SMILES validation category.
7. Save the validated dataset for subsequent molecular feature engineering.

## Expected Output

A CSV file named **drugbank_step13_smiles_validated.csv** containing all DrugBank records together with two additional columns:

- `Valid_SMILES`
- `SMILES_Status`

No DrugBank records are removed during this step. The validation results will be used in the next stage to identify molecules suitable for molecular descriptor and fingerprint generation.

In [17]:
%pip install rdkit


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
# ============================================================
# Step 13 – SMILES Validation
# ============================================================

import pandas as pd
from rdkit import Chem

print("RDKit imported successfully.")

# ------------------------------------------------------------
# Define file paths
# ------------------------------------------------------------

INPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step12_cleaned.csv"

OUTPUT_FILE = "/Users/sankhadeep/Desktop/DrugBank/drugbank_step13_smiles_validated.csv"


# ------------------------------------------------------------
# Load the cleaned dataset
# ------------------------------------------------------------

df = pd.read_csv(
    INPUT_FILE,
    keep_default_na=False
)

print("Dataset Shape:")
print(df.shape)


# ------------------------------------------------------------
# Function to determine SMILES status
# ------------------------------------------------------------

def validate_smiles(smiles):

    # Check whether SMILES is missing or blank
    if not isinstance(smiles, str) or smiles.strip() == "":
        return "Missing"

    # Try to convert the SMILES string into an RDKit molecule
    mol = Chem.MolFromSmiles(smiles)

    # If RDKit successfully creates a molecule
    if mol is not None:
        return "Valid"

    # SMILES exists but RDKit cannot parse it
    return "Invalid"


# ------------------------------------------------------------
# Validate SMILES
# ------------------------------------------------------------

df["SMILES_Status"] = df["SMILES"].apply(validate_smiles)


# ------------------------------------------------------------
# Create Boolean indicator for valid SMILES
# ------------------------------------------------------------

df["Valid_SMILES"] = df["SMILES_Status"].eq("Valid")


# ------------------------------------------------------------
# Count SMILES validation results
# ------------------------------------------------------------

valid_count = (df["SMILES_Status"] == "Valid").sum()

invalid_count = (df["SMILES_Status"] == "Invalid").sum()

missing_count = (df["SMILES_Status"] == "Missing").sum()

total_drugs = len(df)


# ------------------------------------------------------------
# Display validation results
# ------------------------------------------------------------

print("\nSMILES Validation Results")
print("-----------------------------")

print("Total Drugs:",
      total_drugs)

print("Valid SMILES:",
      valid_count)

print("Invalid SMILES:",
      invalid_count)

print("Missing SMILES:",
      missing_count)


# ------------------------------------------------------------
# Calculate percentages
# ------------------------------------------------------------

valid_percentage = (
    valid_count / total_drugs
) * 100

invalid_percentage = (
    invalid_count / total_drugs
) * 100

missing_percentage = (
    missing_count / total_drugs
) * 100


print("\nSMILES Validation Percentage")
print("-----------------------------")

print(
    f"Valid SMILES: {valid_percentage:.2f}%"
)

print(
    f"Invalid SMILES: {invalid_percentage:.2f}%"
)

print(
    f"Missing SMILES: {missing_percentage:.2f}%"
)


# ------------------------------------------------------------
# Create validation summary table
# ------------------------------------------------------------

smiles_summary = pd.DataFrame({

    "SMILES_Status": [
        "Valid",
        "Invalid",
        "Missing"
    ],

    "Count": [
        valid_count,
        invalid_count,
        missing_count
    ]
})


smiles_summary["Percentage"] = (
    smiles_summary["Count"] / total_drugs
) * 100


smiles_summary["Percentage"] = (
    smiles_summary["Percentage"].round(2)
)


print("\nSMILES Validation Summary:")
print(smiles_summary)


# ------------------------------------------------------------
# Verify that all records have been classified
# ------------------------------------------------------------

classified_total = (
    valid_count +
    invalid_count +
    missing_count
)

print("\nValidation Check:")
print("Total classified records:",
      classified_total)

print("Total dataset records:",
      total_drugs)


if classified_total == total_drugs:
    print("All DrugBank records were successfully classified.")
else:
    print("Warning: Some records were not classified.")


# ------------------------------------------------------------
# Preview validated records
# ------------------------------------------------------------

print("\nFirst Five Records:")
print(
    df[
        [
            "DrugBank_ID",
            "Drug_Name",
            "SMILES",
            "SMILES_Status",
            "Valid_SMILES"
        ]
    ].head()
)


# ------------------------------------------------------------
# Save validated dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nValidated Dataset Shape:")
print(df.shape)

print("\nValidated dataset saved at:")
print(OUTPUT_FILE)

RDKit imported successfully.
Dataset Shape:
(19856, 18)


[00:07:44] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[00:07:44] SMILES Parse Error: syntax error while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[00:07:44] SMILES Parse Error: check for mistakes around position 84:
[00:07:44] C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O
[00:07:44] ~~~~~~~~~~~~~~~~~~~~^
[00:07:44] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[00:07:44] SMILES Parse Error: check for mistakes around position 40:
[00:07:44] 1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2
[00:07:44] ~~~~~~~~~~~~~~~~~~~~^
[00:07:44] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C


SMILES Validation Results
-----------------------------
Total Drugs: 19856
Valid SMILES: 14616
Invalid SMILES: 10
Missing SMILES: 5230

SMILES Validation Percentage
-----------------------------
Valid SMILES: 73.61%
Invalid SMILES: 0.05%
Missing SMILES: 26.34%

SMILES Validation Summary:
  SMILES_Status  Count  Percentage
0         Valid  14616       73.61
1       Invalid     10        0.05
2       Missing   5230       26.34

Validation Check:
Total classified records: 19856
Total dataset records: 19856
All DrugBank records were successfully classified.

First Five Records:
  DrugBank_ID            Drug_Name SMILES SMILES_Status  Valid_SMILES
0     DB00001            Lepirudin              Missing         False
1     DB00002            Cetuximab              Missing         False
2     DB00003         Dornase alfa              Missing         False
3     DB00004  Denileukin diftitox              Missing         False
4     DB00005           Etanercept              Missing         Fals

In [19]:
invalid_smiles = df[df["SMILES_Status"] == "Invalid"]

print(invalid_smiles[
    [
        "DrugBank_ID",
        "Drug_Name",
        "SMILES"
    ]
])

      DrugBank_ID                                          Drug_Name  \
1827      DB01929                5-Chloryl-2,4,6-quinazolinetriamine   
1896      DB01999  5,10,15,20-Tetrakis(4-Sulpfonatophenyl)-21h,23...   
2519      DB02667                           Vitamin B 12 factor IIIm   
3858      DB04100        Tricarbonyl(1,10-phenanthroline)rhenium(1+)   
4175      DB04444                           Tetrafluoroaluminate Ion   
7440      DB08276  trifluoro-[hydroxy-[hydroxy-[2-(N-methyl-2-nit...   
8478      DB09385                               Cyanocobalamin Co-57   
9763      DB11630                                         Temoporfin   
10563     DB12453                                   Chlorine dioxide   
15398     DB17408                                        Borocaptate   

                                                  SMILES  
1827              NC1=C(C2=C(N)N=C(N)N=C2C=C1)[Cl](=O)=O  
1896   OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\...  
2519   COC1=CC=C2N(C=NC2=C1)[C

# Step 14 – Molecular Descriptor Generation

## Objective

The objective of this step is to generate numerical molecular descriptors from the valid SMILES representations identified in Step 13.

Molecular descriptors represent important physicochemical and structural properties of a molecule in numerical form. These numerical values can later be used as input features for machine learning models.

Only DrugBank records with valid SMILES representations are used for descriptor calculation. Records with missing or invalid SMILES are retained in the dataset, but their descriptor values remain missing.

The following molecular descriptors are calculated using RDKit:

- **Molecular Weight (MolWt)** – represents the molecular mass of the compound.
- **LogP** – represents the estimated lipophilicity of the molecule.
- **TPSA** – represents the topological polar surface area.
- **Hydrogen Bond Donors (HBD)** – number of groups capable of donating hydrogen bonds.
- **Hydrogen Bond Acceptors (HBA)** – number of groups capable of accepting hydrogen bonds.
- **Rotatable Bonds** – number of rotatable bonds, providing an indication of molecular flexibility.
- **Ring Count** – total number of rings present in the molecular structure.

This step performs the following operations:

1. Load the SMILES-validated dataset generated in Step 13.
2. Convert each valid SMILES representation into an RDKit molecular object.
3. Calculate the selected molecular descriptors.
4. Store the descriptor values as additional columns in the DrugBank dataset.
5. Leave descriptor values missing for drugs with invalid or unavailable SMILES.
6. Check the number of drugs for which molecular descriptors were successfully generated.
7. Save the resulting dataset for subsequent molecular feature analysis and machine learning.

## Expected Output

A CSV file named **drugbank_step14_molecular_descriptors.csv** containing the existing DrugBank information together with the following additional molecular descriptor columns:

- MolWt
- LogP
- TPSA
- HBD
- HBA
- Rotatable_Bonds
- Ring_Count

All original DrugBank records are retained. Molecular descriptor values are generated only for records containing valid SMILES representations.

In [20]:
# ============================================================
# Step 14 – Molecular Descriptor Generation
# ============================================================

import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import rdMolDescriptors


# ------------------------------------------------------------
# Define file paths
# ------------------------------------------------------------

INPUT_FILE = (
    "/Users/sankhadeep/Desktop/DrugBank/"
    "drugbank_step13_smiles_validated.csv"
)

OUTPUT_FILE = (
    "/Users/sankhadeep/Desktop/DrugBank/"
    "drugbank_step14_molecular_descriptors.csv"
)


# ------------------------------------------------------------
# Load Step 13 dataset
# ------------------------------------------------------------

df = pd.read_csv(
    INPUT_FILE,
    keep_default_na=False
)

print("Dataset Shape:")
print(df.shape)


# ------------------------------------------------------------
# Function to calculate molecular descriptors
# ------------------------------------------------------------

def calculate_descriptors(smiles, valid_smiles):

    # Descriptors are calculated only for valid SMILES
    if not valid_smiles:
        return pd.Series([
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        ])

    # Convert SMILES into an RDKit molecule
    mol = Chem.MolFromSmiles(smiles)

    # Additional safety check
    if mol is None:
        return pd.Series([
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        ])

    # Calculate molecular descriptors
    molecular_weight = Descriptors.MolWt(mol)

    logp = Descriptors.MolLogP(mol)

    tpsa = rdMolDescriptors.CalcTPSA(mol)

    hbd = Lipinski.NumHDonors(mol)

    hba = Lipinski.NumHAcceptors(mol)

    rotatable_bonds = Lipinski.NumRotatableBonds(mol)

    ring_count = rdMolDescriptors.CalcNumRings(mol)

    return pd.Series([
        molecular_weight,
        logp,
        tpsa,
        hbd,
        hba,
        rotatable_bonds,
        ring_count
    ])


# ------------------------------------------------------------
# Generate molecular descriptors
# ------------------------------------------------------------

descriptor_columns = [
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Ring_Count"
]


df[descriptor_columns] = df.apply(
    lambda row: calculate_descriptors(
        row["SMILES"],
        row["Valid_SMILES"]
    ),
    axis=1
)


# ------------------------------------------------------------
# Count successfully generated descriptors
# ------------------------------------------------------------

descriptor_count = df["MolWt"].notna().sum()

print("\nMolecular Descriptor Generation")
print("--------------------------------")

print("Total Drugs:",
      len(df))

print("Drugs with Valid SMILES:",
      df["Valid_SMILES"].sum())

print("Drugs with Molecular Descriptors:",
      descriptor_count)

print(
    "Drugs without Molecular Descriptors:",
    len(df) - descriptor_count
)


# ------------------------------------------------------------
# Calculate percentage of drugs with descriptors
# ------------------------------------------------------------

descriptor_percentage = (
    descriptor_count / len(df)
) * 100

print(
    f"\nPercentage with Molecular Descriptors: "
    f"{descriptor_percentage:.2f}%"
)


# ------------------------------------------------------------
# Display descriptor summary statistics
# ------------------------------------------------------------

print("\nMolecular Descriptor Summary Statistics:")
print(
    df[descriptor_columns].describe()
)


# ------------------------------------------------------------
# Preview generated descriptors
# ------------------------------------------------------------

print("\nFirst Five Drugs with Molecular Descriptors:")

print(
    df.loc[
        df["Valid_SMILES"],
        [
            "DrugBank_ID",
            "Drug_Name",
            "MolWt",
            "LogP",
            "TPSA",
            "HBD",
            "HBA",
            "Rotatable_Bonds",
            "Ring_Count"
        ]
    ].head()
)


# ------------------------------------------------------------
# Verify descriptor generation
# ------------------------------------------------------------

valid_smiles_count = df["Valid_SMILES"].sum()

print("\nDescriptor Validation Check:")
print("Valid SMILES:", valid_smiles_count)
print("Descriptor Records:", descriptor_count)


if descriptor_count == valid_smiles_count:
    print(
        "Descriptors were successfully generated "
        "for all valid SMILES."
    )
else:
    print(
        "Warning: Some valid SMILES did not produce "
        "molecular descriptors."
    )


# ------------------------------------------------------------
# Save dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset saved at:")
print(OUTPUT_FILE)

Dataset Shape:
(19856, 20)


[00:07:47] Unusual charge on atom 0 number of radical electrons set to zero
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors
[00:07:49] WARNING: not removing hydrogen atom without neighbors



Molecular Descriptor Generation
--------------------------------
Total Drugs: 19856
Drugs with Valid SMILES: 14616
Drugs with Molecular Descriptors: 14616
Drugs without Molecular Descriptors: 5240

Percentage with Molecular Descriptors: 73.61%

Molecular Descriptor Summary Statistics:
              MolWt          LogP          TPSA           HBD           HBA  \
count  14616.000000  14616.000000  14616.000000  14616.000000  14616.000000   
mean     387.971819      2.103196    101.558835      2.494663      5.338191   
std      263.859949      3.188848    105.727025      3.384936      4.667496   
min        2.016000    -35.163200      0.000000      0.000000      0.000000   
25%      257.252703      0.603525     52.900000      1.000000      3.000000   
50%      350.490000      2.481645     82.265000      2.000000      4.000000   
75%      450.433250      3.940975    115.730000      3.000000      6.000000   
max     6685.037000     57.751000   3071.500000     66.000000    149.000000   

 

In [21]:
descriptor_count = df["MolWt"].notna().sum()

descriptor_count

np.int64(14616)

In [22]:
%pip install pyarrow


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 15 – Loading Open Targets Clinical Indication Data

To support the drug repurposing analysis, clinical indication data from the Open Targets Platform was loaded. This dataset contains drug–disease relationships and includes the ChEMBL drug identifier (`drugId`), disease identifier (`diseaseId`), maximum clinical stage (`maxClinicalStage`), and associated clinical report identifiers (`clinicalReportIds`).

The purpose of this step is to inspect the Open Targets clinical indication dataset before further preprocessing and integration with the DrugBank data.

The Parquet file is loaded using Pandas, and its dimensions and first five records are displayed to verify that the dataset has been read successfully.

In [23]:
import pandas as pd
import pyarrow

print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

clinical_indication_path = "OpenTargets/clinical_indication/clinical_indication.parquet"

clinical_indication = pd.read_parquet(clinical_indication_path)

print("Shape:", clinical_indication.shape)
clinical_indication.head()

Pandas: 2.3.3
PyArrow: 25.0.1
Shape: (86468, 5)


,id,maxClinicalStage,clinicalReportIds,drugId,diseaseId
0,bcae49c2072f6bce9286dcced5374756568bbc17c3c389...,APPROVAL,"[00011703-bc55-4c0c-858c-149dc674bc3c, 006095a...",CHEMBL4,EFO_0000544
1,df2ffa13f012d1eb1938575935f121d406e68257746740...,APPROVAL,[0002ad27-779d-42ab-83b5-bc65453412a1],CHEMBL2105741,MONDO_0008903
2,ae47cca0541b69749b687cb246f1dc942734d45d87400e...,APPROVAL,"[0002eded-a283-4590-a78c-b6ff2027e684, 2ef99c3...",CHEMBL940,HP_0001250
3,af0953f786255487c4dc3852c77c7e1e756f21f9e53758...,APPROVAL,"[0002eded-a283-4590-a78c-b6ff2027e684, 01aafb0...",CHEMBL940,MONDO_0005027
4,8b1223c9af56ff52a4603bacc2c591f65124974a5a5407...,APPROVAL,"[0003458f-352a-46fa-9d99-230daa76ae29, 07ad436...",CHEMBL1481,MONDO_0005148


## Step 16 – Exploring Open Targets Clinical Indication Data

Before integrating the Open Targets data with the DrugBank dataset, the clinical indication data is examined to understand its structure and coverage.

This step investigates:
- the number of unique drugs,
- the number of unique diseases,
- the available clinical stages,
- missing values,
- and duplicate drug–disease pairs.

This analysis helps determine how the Open Targets clinical indication data can be used for constructing drug–disease associations for the subsequent drug repurposing analysis.

In [24]:
# Basic information about the Open Targets clinical indication dataset

print("Dataset shape:", clinical_indication.shape)

print("\nNumber of unique drugs:",
      clinical_indication["drugId"].nunique())

print("Number of unique diseases:",
      clinical_indication["diseaseId"].nunique())

# Missing values
print("\nMissing values:")
print(clinical_indication.isnull().sum())

# Clinical stage distribution
print("\nClinical stage distribution:")
print(clinical_indication["maxClinicalStage"].value_counts(dropna=False))

# Number of unique drug-disease pairs
unique_pairs = clinical_indication[
    ["drugId", "diseaseId"]
].drop_duplicates()

print("\nNumber of unique drug-disease pairs:",
      len(unique_pairs))

# Check duplicate drug-disease pairs
duplicate_pairs = clinical_indication.duplicated(
    subset=["drugId", "diseaseId"]
).sum()

print("Duplicate drug-disease pairs:",
      duplicate_pairs)

Dataset shape: (86468, 5)

Number of unique drugs: 11364
Number of unique diseases: 3749

Missing values:
id                   0
maxClinicalStage     0
clinicalReportIds    0
drugId               0
diseaseId            0
dtype: int64

Clinical stage distribution:
maxClinicalStage
PHASE_2          28370
PHASE_3          15850
APPROVAL         11175
UNKNOWN          10218
PHASE_1           9637
PHASE_1_2         6443
PHASE_2_3         3134
EARLY_PHASE_1     1284
IND                198
PRECLINICAL        111
PREAPPROVAL         48
Name: count, dtype: int64

Number of unique drug-disease pairs: 86468
Duplicate drug-disease pairs: 0


## Step 17 – Investigating Open Targets Drug and Disease Identifiers

The Open Targets clinical indication dataset contains 86,468 unique drug–disease pairs involving 11,364 unique drugs and 3,749 unique diseases, with no missing values or duplicate drug–disease pairs.

Before integrating these associations with the DrugBank dataset, the identifier systems used for drugs and diseases are examined. Open Targets represents drugs using ChEMBL identifiers, while diseases may be represented using different ontology identifiers such as EFO, MONDO, and HP.

This step examines the formats and frequencies of these identifiers to determine the appropriate strategy for subsequent data integration.

In [25]:
# Inspect example drug and disease identifiers

print("Example drug IDs:")
print(clinical_indication["drugId"].drop_duplicates().head(20).to_list())

print("\nExample disease IDs:")
print(clinical_indication["diseaseId"].drop_duplicates().head(20).to_list())


# Extract identifier prefixes
drug_prefix = (
    clinical_indication["drugId"]
    .str.extract(r"^([A-Za-z]+)", expand=False)
)

disease_prefix = (
    clinical_indication["diseaseId"]
    .str.extract(r"^([^_]+)", expand=False)
)


print("\nDrug ID prefix distribution:")
print(drug_prefix.value_counts(dropna=False))

print("\nDisease ID prefix distribution:")
print(disease_prefix.value_counts(dropna=False))

Example drug IDs:
['CHEMBL4', 'CHEMBL2105741', 'CHEMBL940', 'CHEMBL1481', 'CHEMBL1083385', 'CHEMBL580', 'CHEMBL3184512', 'CHEMBL1200384', 'CHEMBL1069', 'CHEMBL3137301', 'CHEMBL1200890', 'CHEMBL112', 'CHEMBL64', 'CHEMBL635', 'CHEMBL1200781', 'CHEMBL1528', 'CHEMBL1697', 'CHEMBL1082', 'CHEMBL1200930', 'CHEMBL1201010']

Example disease IDs:
['EFO_0000544', 'MONDO_0008903', 'HP_0001250', 'MONDO_0005027', 'MONDO_0005148', 'MONDO_0005098', 'MONDO_0005068', 'MONDO_0005542', 'MONDO_0005386', 'MONDO_0002009', 'MONDO_0005618', 'MONDO_0024355', 'EFO_1001753', 'HP_0100806', 'MONDO_1010128', 'MONDO_0005246', 'MONDO_0006887', 'MONDO_0000918', 'MONDO_0005249', 'MONDO_0005242']

Drug ID prefix distribution:
drugId
CHEMBL    86468
Name: count, dtype: int64

Disease ID prefix distribution:
diseaseId
MONDO       70521
HP           8591
EFO          6300
Orphanet      432
MP            430
GO            154
OBA            35
NCIT            3
OTAR            2
Name: count, dtype: int64


# Step 18 – Extracting DrugBank–ChEMBL Identifier Mapping

## Objective

The DrugBank dataset and the Open Targets clinical indication dataset use different identifiers for drugs. DrugBank uses identifiers such as `DB00001`, whereas the Open Targets clinical indication dataset uses ChEMBL identifiers such as `CHEMBL4`.

To integrate the molecular and biological features extracted from DrugBank with the drug–disease associations from Open Targets, a common drug identifier is required.

The DrugBank XML file contains external database identifiers for many drugs. In this step, the `<external-identifiers>` section of each top-level DrugBank drug record is examined to identify entries where the resource is `ChEMBL`.

For each DrugBank drug, the following information is extracted:

- DrugBank identifier
- ChEMBL identifier, where available

A DrugBank–ChEMBL mapping table is then created. The extracted ChEMBL identifiers are also compared with the ChEMBL drug identifiers present in the Open Targets clinical indication dataset.

This mapping will provide the link required for subsequent integration of DrugBank molecular features with Open Targets drug–disease associations.

In [29]:
# ============================================================
# Step 18 – Extract DrugBank–ChEMBL Identifier Mapping
# ============================================================

import xml.etree.ElementTree as ET
import pandas as pd


# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------

XML_FILE = "/Users/sankhadeep/Desktop/DrugBank/full database.xml"

OUTPUT_FILE = (
    "/Users/sankhadeep/Desktop/DrugBank/"
    "drugbank_chembl_mapping.csv"
)


# ------------------------------------------------------------
# DrugBank XML namespace
# ------------------------------------------------------------

NS = "{http://www.drugbank.ca}"


# ------------------------------------------------------------
# Store extracted mappings
# ------------------------------------------------------------

mapping_records = []


# ------------------------------------------------------------
# Parse DrugBank XML
# ------------------------------------------------------------

context = ET.iterparse(
    XML_FILE,
    events=("start", "end")
)

depth = 0


for event, elem in context:

    # --------------------------------------------------------
    # Track XML depth
    # --------------------------------------------------------

    if event == "start":
        depth += 1


    elif event == "end":

        # ----------------------------------------------------
        # Process only top-level <drug> records
        # ----------------------------------------------------

        if elem.tag == NS + "drug" and depth == 2:

            # ------------------------------------------------
            # Extract PRIMARY DrugBank ID
            # ------------------------------------------------

            primary_drugbank_id = elem.find(
                NS + "drugbank-id[@primary='true']"
            )

            if (
                primary_drugbank_id is not None
                and primary_drugbank_id.text
            ):

                drugbank_id = (
                    primary_drugbank_id.text.strip()
                )

            else:
                drugbank_id = ""


            # ------------------------------------------------
            # Extract ChEMBL ID
            # ------------------------------------------------

            chembl_id = ""

            external_identifiers = elem.find(
                NS + "external-identifiers"
            )

            if external_identifiers is not None:

                for external_id in external_identifiers.findall(
                    NS + "external-identifier"
                ):

                    resource = external_id.findtext(
                        NS + "resource",
                        default=""
                    ).strip()

                    identifier = external_id.findtext(
                        NS + "identifier",
                        default=""
                    ).strip()


                    # ----------------------------------------
                    # Identify ChEMBL external identifier
                    # ----------------------------------------

                    if resource.lower() == "chembl":

                        chembl_id = identifier.strip()

                        break


            # ------------------------------------------------
            # Store DrugBank–ChEMBL mapping
            # ------------------------------------------------

            mapping_records.append(
                [
                    drugbank_id,
                    chembl_id
                ]
            )


            # ------------------------------------------------
            # Release memory
            # ------------------------------------------------

            elem.clear()


        # ----------------------------------------------------
        # Decrease XML depth after processing end event
        # ----------------------------------------------------

        depth -= 1


# ============================================================
# Create DataFrame
# ============================================================

drugbank_chembl = pd.DataFrame(
    mapping_records,
    columns=[
        "DrugBank_ID",
        "ChEMBL_ID"
    ]
)


# ------------------------------------------------------------
# Clean identifier columns
# ------------------------------------------------------------

drugbank_chembl["DrugBank_ID"] = (
    drugbank_chembl["DrugBank_ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)

drugbank_chembl["ChEMBL_ID"] = (
    drugbank_chembl["ChEMBL_ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ============================================================
# Basic DrugBank–ChEMBL Mapping Statistics
# ============================================================

total_drugs = len(drugbank_chembl)


mapped_drugs = (
    drugbank_chembl["ChEMBL_ID"]
    .ne("")
    .sum()
)


unmapped_drugs = (
    total_drugs
    - mapped_drugs
)


unique_chembl_ids = (
    drugbank_chembl.loc[
        drugbank_chembl["ChEMBL_ID"].ne(""),
        "ChEMBL_ID"
    ]
    .nunique()
)


print("DrugBank–ChEMBL mapping statistics")
print("----------------------------------")

print(
    "Total DrugBank records:",
    total_drugs
)

print(
    "DrugBank records with ChEMBL ID:",
    mapped_drugs
)

print(
    "DrugBank records without ChEMBL ID:",
    unmapped_drugs
)

print(
    "Unique ChEMBL IDs in DrugBank:",
    unique_chembl_ids
)


# ------------------------------------------------------------
# Calculate mapping percentage
# ------------------------------------------------------------

if total_drugs > 0:

    mapping_percentage = (
        mapped_drugs
        / total_drugs
        * 100
    )

else:

    mapping_percentage = 0


print(
    "Percentage of DrugBank records mapped to ChEMBL:",
    f"{mapping_percentage:.2f}%"
)


# ============================================================
# Display Example Mappings
# ============================================================

print(
    "\nExample DrugBank–ChEMBL mappings:"
)


display(
    drugbank_chembl[
        drugbank_chembl["ChEMBL_ID"].ne("")
    ]
    .head(10)
)


# ============================================================
# Save DrugBank–ChEMBL Mapping Table
# ============================================================

drugbank_chembl.to_csv(
    OUTPUT_FILE,
    index=False
)


print(
    "\nMapping saved to:"
)

print(
    OUTPUT_FILE
)


# ============================================================
# Compare DrugBank ChEMBL IDs with Open Targets
# ============================================================

# ------------------------------------------------------------
# Extract unique ChEMBL IDs from DrugBank
# ------------------------------------------------------------

drugbank_chembl_ids = set(
    drugbank_chembl.loc[
        drugbank_chembl["ChEMBL_ID"].ne(""),
        "ChEMBL_ID"
    ]
)


# ------------------------------------------------------------
# Clean Open Targets drug IDs
# ------------------------------------------------------------

open_targets_drug_ids_clean = (
    clinical_indication["drugId"]
    .dropna()
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Remove blank IDs
# ------------------------------------------------------------

open_targets_drug_ids_clean = (
    open_targets_drug_ids_clean[
        open_targets_drug_ids_clean.ne("")
    ]
)


# ------------------------------------------------------------
# Convert to unique set
# ------------------------------------------------------------

open_targets_chembl_ids = set(
    open_targets_drug_ids_clean
)


# ------------------------------------------------------------
# Find common ChEMBL identifiers
# ------------------------------------------------------------

common_chembl_ids = (
    drugbank_chembl_ids
    .intersection(
        open_targets_chembl_ids
    )
)


# ============================================================
# Open Targets Comparison Statistics
# ============================================================

print(
    "\nOpen Targets comparison"
)

print(
    "----------------------------------"
)


print(
    "Unique ChEMBL IDs in DrugBank mapping:",
    len(drugbank_chembl_ids)
)


print(
    "Unique drugs in Open Targets:",
    len(open_targets_chembl_ids)
)


print(
    "Common ChEMBL IDs:",
    len(common_chembl_ids)
)


print(
    "Open Targets drugs matched to DrugBank:",
    len(common_chembl_ids)
)


# ------------------------------------------------------------
# Calculate percentage of Open Targets drugs matched
# ------------------------------------------------------------

if len(open_targets_chembl_ids) > 0:

    open_targets_match_percentage = (
        len(common_chembl_ids)
        / len(open_targets_chembl_ids)
        * 100
    )

else:

    open_targets_match_percentage = 0


print(
    "Percentage of Open Targets drugs matched to DrugBank:",
    f"{open_targets_match_percentage:.2f}%"
)


# ------------------------------------------------------------
# Calculate percentage of DrugBank ChEMBL IDs found in
# Open Targets
# ------------------------------------------------------------

if len(drugbank_chembl_ids) > 0:

    drugbank_match_percentage = (
        len(common_chembl_ids)
        / len(drugbank_chembl_ids)
        * 100
    )

else:

    drugbank_match_percentage = 0


print(
    "Percentage of DrugBank ChEMBL IDs found in Open Targets:",
    f"{drugbank_match_percentage:.2f}%"
)


# ============================================================
# Display Example Common IDs
# ============================================================

print(
    "\nExample ChEMBL IDs present in both DrugBank "
    "and Open Targets:"
)


example_common_ids = sorted(
    common_chembl_ids
)[:10]


for chembl_id in example_common_ids:

    print(
        chembl_id
    )

DrugBank–ChEMBL mapping statistics
----------------------------------
Total DrugBank records: 19856
DrugBank records with ChEMBL ID: 11035
DrugBank records without ChEMBL ID: 8821
Unique ChEMBL IDs in DrugBank: 11035
Percentage of DrugBank records mapped to ChEMBL: 55.58%

Example DrugBank–ChEMBL mappings:


,DrugBank_ID,ChEMBL_ID
0,DB00001,CHEMBL1201666
1,DB00002,CHEMBL1201577
2,DB00003,CHEMBL1201431
3,DB00004,CHEMBL1201550
4,DB00005,CHEMBL1201572
5,DB00006,CHEMBL2103749
6,DB00007,CHEMBL1201199
7,DB00008,CHEMBL1201560
8,DB00009,CHEMBL1201593
10,DB00011,CHEMBL2108509



Mapping saved to:
/Users/sankhadeep/Desktop/DrugBank/drugbank_chembl_mapping.csv

Open Targets comparison
----------------------------------
Unique ChEMBL IDs in DrugBank mapping: 11035
Unique drugs in Open Targets: 11364
Common ChEMBL IDs: 5406
Open Targets drugs matched to DrugBank: 5406
Percentage of Open Targets drugs matched to DrugBank: 47.57%
Percentage of DrugBank ChEMBL IDs found in Open Targets: 48.99%

Example ChEMBL IDs present in both DrugBank and Open Targets:
CHEMBL100
CHEMBL1000
CHEMBL100014
CHEMBL100116
CHEMBL1002
CHEMBL100259
CHEMBL1004
CHEMBL1005
CHEMBL1006
CHEMBL1008


# Step 19 – Integrating DrugBank Features with Open Targets Drug–Disease Associations

## Objective

The objective of this step is to integrate the DrugBank dataset with the Open Targets clinical indication dataset using the DrugBank–ChEMBL identifier mapping created in Step 18.

The DrugBank dataset contains molecular, biological, and therapeutic information for each drug, including SMILES representations and molecular descriptors generated using RDKit. DrugBank primarily uses DrugBank identifiers such as `DB00001`.

In contrast, the Open Targets clinical indication dataset represents drugs using ChEMBL identifiers and provides drug–disease associations through the `drugId` and `diseaseId` columns.

Step 18 established a mapping between primary DrugBank identifiers and ChEMBL identifiers. This mapping is therefore used as a bridge between the two datasets.

The integration is performed in two main stages:

1. The DrugBank molecular-feature dataset is joined with the DrugBank–ChEMBL mapping using `DrugBank_ID`.
2. The resulting dataset is joined with the Open Targets clinical indication dataset by matching the DrugBank `ChEMBL_ID` with the Open Targets `drugId`.

The resulting integrated dataset contains rows representing matched drug–disease associations. Consequently, the same drug may appear in multiple rows when it is associated with multiple diseases.

This step also examines:

- the number of matched drug–disease association rows,
- the number of unique matched drugs,
- the number of unique diseases,
- the clinical-stage distribution,
- duplicate drug–disease pairs,
- and the availability of molecular descriptors among the matched associations.

The integrated dataset will provide the basis for the subsequent construction of the machine-learning dataset for drug repurposing.

In [30]:
# ============================================================
# Step 19 – Integrate DrugBank Features with Open Targets
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Output file
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/Users/sankhadeep/Desktop/DrugBank/"
    "drugbank_opentargets_integrated.csv"
)


# ============================================================
# Step 19.1
# Add ChEMBL identifiers to the DrugBank feature dataset
# ============================================================

drugbank_with_chembl = df.merge(
    drugbank_chembl,
    on="DrugBank_ID",
    how="left"
)


# ------------------------------------------------------------
# Clean identifier columns after merge
# ------------------------------------------------------------

drugbank_with_chembl["DrugBank_ID"] = (
    drugbank_with_chembl["DrugBank_ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)

drugbank_with_chembl["ChEMBL_ID"] = (
    drugbank_with_chembl["ChEMBL_ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# DrugBank + ChEMBL mapping statistics
# ------------------------------------------------------------

print("DrugBank dataset after adding ChEMBL IDs:")
print("Shape:", drugbank_with_chembl.shape)


drugs_with_chembl = (
    drugbank_with_chembl["ChEMBL_ID"]
    .ne("")
    .sum()
)


drugs_without_chembl = (
    drugbank_with_chembl["ChEMBL_ID"]
    .eq("")
    .sum()
)


print(
    "Drugs with ChEMBL ID:",
    drugs_with_chembl
)

print(
    "Drugs without ChEMBL ID:",
    drugs_without_chembl
)


# ============================================================
# Step 19.2
# Clean Open Targets drug identifiers
# ============================================================

clinical_indication = clinical_indication.copy()


clinical_indication["drugId"] = (
    clinical_indication["drugId"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Remove blank Open Targets drug IDs
# ------------------------------------------------------------

clinical_indication = clinical_indication[
    clinical_indication["drugId"].ne("")
].copy()


# ------------------------------------------------------------
# Create cleaned unique Open Targets ChEMBL ID set
# ------------------------------------------------------------

open_targets_chembl_ids = set(
    clinical_indication["drugId"]
)


# ============================================================
# Step 19.3
# Keep DrugBank records that can be linked to Open Targets
# ============================================================

drugbank_matched = drugbank_with_chembl[
    drugbank_with_chembl["ChEMBL_ID"].isin(
        open_targets_chembl_ids
    )
].copy()


print(
    "\nDrugBank–Open Targets matched drugs"
)

print(
    "------------------------------------"
)


print(
    "Matched DrugBank records:",
    len(drugbank_matched)
)


print(
    "Unique matched DrugBank IDs:",
    drugbank_matched[
        "DrugBank_ID"
    ].nunique()
)


print(
    "Unique matched ChEMBL IDs:",
    drugbank_matched[
        "ChEMBL_ID"
    ].nunique()
)


# ============================================================
# Step 19.4
# Merge DrugBank features with Open Targets clinical indications
# ============================================================

integrated_df = drugbank_matched.merge(
    clinical_indication,
    left_on="ChEMBL_ID",
    right_on="drugId",
    how="inner"
)


# ============================================================
# Integrated Dataset Statistics
# ============================================================

print(
    "\nIntegrated Dataset"
)

print(
    "-----------------------------"
)


print(
    "Shape:",
    integrated_df.shape
)


print(
    "Number of integrated rows:",
    len(integrated_df)
)


print(
    "Unique DrugBank drugs:",
    integrated_df[
        "DrugBank_ID"
    ].nunique()
)


print(
    "Unique ChEMBL drugs:",
    integrated_df[
        "ChEMBL_ID"
    ].nunique()
)


print(
    "Unique diseases:",
    integrated_df[
        "diseaseId"
    ].nunique()
)


# ============================================================
# Check consistency with Step 18
# ============================================================

step18_common_count = len(
    common_chembl_ids
)

step19_common_count = (
    integrated_df[
        "ChEMBL_ID"
    ].nunique()
)


print(
    "\nStep 18 / Step 19 consistency check"
)

print(
    "------------------------------------"
)


print(
    "Common ChEMBL IDs found in Step 18:",
    step18_common_count
)


print(
    "Unique ChEMBL IDs retained after Step 19 integration:",
    step19_common_count
)


difference = (
    step18_common_count
    - step19_common_count
)


print(
    "Difference:",
    difference
)


if difference == 0:

    print(
        "All Step 18 matched ChEMBL IDs are present "
        "in the integrated dataset."
    )

else:

    print(
        "Some Step 18 matched ChEMBL IDs are not present "
        "in the DrugBank feature dataset df."
    )


# ============================================================
# Check duplicate DrugBank–Disease pairs
# ============================================================

duplicate_pairs = integrated_df.duplicated(
    subset=[
        "DrugBank_ID",
        "diseaseId"
    ]
).sum()


print(
    "\nDuplicate DrugBank–disease pairs:",
    duplicate_pairs
)


# ------------------------------------------------------------
# Count unique DrugBank–disease pairs
# ------------------------------------------------------------

unique_drug_disease_pairs = (
    integrated_df[
        [
            "DrugBank_ID",
            "diseaseId"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


print(
    "Unique DrugBank–disease pairs:",
    unique_drug_disease_pairs
)


# ============================================================
# Clinical Stage Distribution
# ============================================================

print(
    "\nClinical Stage Distribution:"
)


print(
    integrated_df[
        "maxClinicalStage"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


# ============================================================
# Molecular Descriptor Availability
# ============================================================

descriptor_columns = [
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Ring_Count"
]


print(
    "\nMolecular Descriptor Availability"
)

print(
    "----------------------------------"
)


for column in descriptor_columns:

    # Check that the descriptor exists
    if column in integrated_df.columns:

        available = (
            integrated_df[
                column
            ]
            .notna()
            .sum()
        )


        if len(integrated_df) > 0:

            percentage = (
                available
                / len(integrated_df)
                * 100
            )

        else:

            percentage = 0


        print(
            f"{column}: "
            f"{available} "
            f"({percentage:.2f}%)"
        )

    else:

        print(
            f"{column}: column not found"
        )


# ============================================================
# Number of Unique Drugs with Molecular Descriptors
# ============================================================

if "MolWt" in integrated_df.columns:

    drugs_with_descriptors = (
        integrated_df.loc[
            integrated_df[
                "MolWt"
            ].notna(),
            "DrugBank_ID"
        ]
        .nunique()
    )

else:

    drugs_with_descriptors = 0


print(
    "\nUnique matched drugs with molecular descriptors:",
    drugs_with_descriptors
)


# ============================================================
# Number of Unique Drug–Disease Pairs with Molecular
# Descriptors
# ============================================================

if "MolWt" in integrated_df.columns:

    pairs_with_descriptors = (
        integrated_df.loc[
            integrated_df[
                "MolWt"
            ].notna(),
            [
                "DrugBank_ID",
                "diseaseId"
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )

else:

    pairs_with_descriptors = 0


print(
    "Unique drug–disease pairs with molecular descriptors:",
    pairs_with_descriptors
)


# ============================================================
# Preview Important Columns
# ============================================================

preview_columns = [
    "DrugBank_ID",
    "Drug_Name",
    "ChEMBL_ID",
    "diseaseId",
    "maxClinicalStage",
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Ring_Count"
]


# ------------------------------------------------------------
# Keep only preview columns that actually exist
# ------------------------------------------------------------

available_preview_columns = [
    column
    for column in preview_columns
    if column in integrated_df.columns
]


print(
    "\nFirst 10 integrated drug–disease associations:"
)


display(
    integrated_df[
        available_preview_columns
    ].head(10)
)


# ============================================================
# Save Integrated Dataset
# ============================================================

integrated_df.to_csv(
    OUTPUT_FILE,
    index=False
)


print(
    "\nIntegrated dataset saved to:"
)

print(
    OUTPUT_FILE
)

DrugBank dataset after adding ChEMBL IDs:
Shape: (19856, 28)
Drugs with ChEMBL ID: 11035
Drugs without ChEMBL ID: 8821

DrugBank–Open Targets matched drugs
------------------------------------
Matched DrugBank records: 5406
Unique matched DrugBank IDs: 5406
Unique matched ChEMBL IDs: 5406

Integrated Dataset
-----------------------------
Shape: (56044, 33)
Number of integrated rows: 56044
Unique DrugBank drugs: 5406
Unique ChEMBL drugs: 5406
Unique diseases: 3445

Step 18 / Step 19 consistency check
------------------------------------
Common ChEMBL IDs found in Step 18: 5406
Unique ChEMBL IDs retained after Step 19 integration: 5406
Difference: 0
All Step 18 matched ChEMBL IDs are present in the integrated dataset.

Duplicate DrugBank–disease pairs: 0
Unique DrugBank–disease pairs: 56044

Clinical Stage Distribution:
maxClinicalStage
APPROVAL          7171
EARLY_PHASE_1      951
IND                 85
PHASE_1           5469
PHASE_1_2         3725
PHASE_2          18772
PHASE_2_3      

,DrugBank_ID,Drug_Name,ChEMBL_ID,diseaseId,maxClinicalStage,MolWt,LogP,TPSA,HBD,HBA,Rotatable_Bonds,Ring_Count
0,DB00001,Lepirudin,CHEMBL1201666,HP_0004419,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DB00001,Lepirudin,CHEMBL1201666,HP_0001873,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DB00001,Lepirudin,CHEMBL1201666,HP_0001907,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DB00001,Lepirudin,CHEMBL1201666,MONDO_0008433,PHASE_1_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DB00002,Cetuximab,CHEMBL1201577,MONDO_0021310,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,DB00002,Cetuximab,CHEMBL1201577,MONDO_0005096,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,DB00002,Cetuximab,CHEMBL1201577,MONDO_0005008,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DB00002,Cetuximab,CHEMBL1201577,MONDO_0024331,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,DB00002,Cetuximab,CHEMBL1201577,MONDO_0005575,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DB00002,Cetuximab,CHEMBL1201577,MONDO_0005586,APPROVAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Integrated dataset saved to:
/Users/sankhadeep/Desktop/DrugBank/drugbank_opentargets_integrated.csv


In [31]:
print("Available variables:")
%whos

Available variables:
Variable                        Type                 Data/Info
--------------------------------------------------------------
Chem                            module               <module 'rdkit.Chem' from<...>/rdkit/Chem/__init__.py'>
Descriptors                     module               <module 'rdkit.Chem.Descr<...>kit/Chem/Descriptors.py'>
ET                              module               <module 'xml.etree.Elemen<...>ml/etree/ElementTree.py'>
FILE_PATH                       str                  /Users/sankhadeep/Desktop<...>nk/drugbank_step9_atc.csv
INPUT_FILE                      str                  /Users/sankhadeep/Desktop<...>ep13_smiles_validated.csv
Lipinski                        module               <module 'rdkit.Chem.Lipin<...>/rdkit/Chem/Lipinski.py'>
NS                              str                  {http://www.drugbank.ca}
OUTPUT_FILE                     str                  /Users/sankhadeep/Desktop<...>pentargets_integrated.csv
XML_FILE    

# Step 20 – Preparing the Integrated Drug–Disease Dataset for Machine Learning

## Objective

The objective of this step is to prepare the integrated DrugBank–Open Targets dataset for subsequent machine-learning dataset construction.

The integrated dataset produced in Step 19 contains matched DrugBank–Open Targets drug–disease associations. Before constructing positive and negative examples for supervised machine learning, the quality and completeness of this integrated dataset must be assessed.

This step therefore:

1. checks whether duplicate `DrugBank_ID`–`diseaseId` pairs are present,
2. verifies the uniqueness of the drug–disease associations,
3. examines missing molecular-descriptor values,
4. identifies associations for which all required molecular descriptors are available,
5. creates a descriptor-complete subset for later machine-learning analysis,
6. examines the clinical-stage distribution of retained associations,
7. compares the number of drugs, diseases, and associations before and after descriptor filtering.

The molecular descriptors generated from DrugBank SMILES representations include:

- `MolWt` – molecular weight,
- `LogP` – octanol/water partition coefficient,
- `TPSA` – topological polar surface area,
- `HBD` – hydrogen-bond donors,
- `HBA` – hydrogen-bond acceptors,
- `Rotatable_Bonds` – number of rotatable bonds,
- `Ring_Count` – number of molecular rings.

Rows without complete molecular descriptors are not treated as negative drug–disease associations. They are excluded only from the descriptor-based modelling subset because the required predictor variables are unavailable.

The resulting descriptor-complete dataset will form the drug-side component of the supervised machine-learning dataset. Disease-related features and positive/negative class construction will be addressed in subsequent steps.

In [32]:
# ============================================================
# Step 20 – Prepare Integrated Drug–Disease Dataset
#            for Machine Learning
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Output file
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/Users/sankhadeep/Desktop/DrugBank/"
    "drugbank_opentargets_descriptor_complete.csv"
)


# ============================================================
# Step 20.1
# Create a working copy of the integrated dataset
# ============================================================

association_df = integrated_df.copy()


print("Initial integrated dataset")
print("--------------------------")

print("Shape:", association_df.shape)

print(
    "Number of association rows:",
    len(association_df)
)

print(
    "Unique DrugBank drugs:",
    association_df["DrugBank_ID"].nunique()
)

print(
    "Unique ChEMBL drugs:",
    association_df["ChEMBL_ID"].nunique()
)

print(
    "Unique diseases:",
    association_df["diseaseId"].nunique()
)


# ============================================================
# Step 20.2
# Check for duplicate DrugBank–Disease pairs
# ============================================================

duplicate_mask = association_df.duplicated(
    subset=[
        "DrugBank_ID",
        "diseaseId"
    ],
    keep=False
)


duplicate_rows = association_df[
    duplicate_mask
].copy()


duplicate_pair_count = (
    association_df
    .duplicated(
        subset=[
            "DrugBank_ID",
            "diseaseId"
        ]
    )
    .sum()
)


print("\nDrug–Disease Pair Validation")
print("----------------------------")

print(
    "Duplicate DrugBank–disease rows:",
    duplicate_pair_count
)


# ------------------------------------------------------------
# Count unique DrugBank–disease pairs
# ------------------------------------------------------------

unique_pair_count = (
    association_df[
        [
            "DrugBank_ID",
            "diseaseId"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


print(
    "Unique DrugBank–disease pairs:",
    unique_pair_count
)


# ------------------------------------------------------------
# Display duplicated pairs if any are found
# ------------------------------------------------------------

if len(duplicate_rows) == 0:

    print(
        "No duplicated DrugBank_ID–diseaseId "
        "pairs were found."
    )

else:

    print(
        "\nDuplicated DrugBank_ID–diseaseId "
        "pairs were found."
    )

    print(
        "Number of rows involved in duplicated pairs:",
        len(duplicate_rows)
    )

    duplicate_rows = duplicate_rows.sort_values(
        [
            "DrugBank_ID",
            "diseaseId"
        ]
    )

    display(
        duplicate_rows[
            [
                column
                for column in [
                    "DrugBank_ID",
                    "Drug_Name",
                    "ChEMBL_ID",
                    "diseaseId",
                    "maxClinicalStage"
                ]
                if column in duplicate_rows.columns
            ]
        ].head(20)
    )


# ============================================================
# Step 20.3
# Define molecular descriptor columns
# ============================================================

descriptor_columns = [
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Ring_Count"
]


# ============================================================
# Step 20.4
# Check whether all required descriptor columns exist
# ============================================================

missing_descriptor_columns = [
    column
    for column in descriptor_columns
    if column not in association_df.columns
]


print("\nMolecular Descriptor Column Check")
print("---------------------------------")


if len(missing_descriptor_columns) == 0:

    print(
        "All required molecular descriptor "
        "columns are present."
    )

else:

    print(
        "Missing molecular descriptor columns:"
    )

    print(
        missing_descriptor_columns
    )


# ============================================================
# Step 20.5
# Examine missing values in each molecular descriptor
# ============================================================

print("\nMolecular Descriptor Missing Values")
print("-----------------------------------")


for column in descriptor_columns:

    if column in association_df.columns:

        missing_count = (
            association_df[column]
            .isna()
            .sum()
        )

        missing_percentage = (
            missing_count
            / len(association_df)
            * 100
        )

        print(
            f"{column}: "
            f"{missing_count} missing "
            f"({missing_percentage:.2f}%)"
        )


# ============================================================
# Step 20.6
# Identify rows with complete molecular descriptors
# ============================================================

association_df[
    "Has_All_Descriptors"
] = (
    association_df[
        descriptor_columns
    ]
    .notna()
    .all(axis=1)
)


# ------------------------------------------------------------
# Identify rows where all descriptors are missing
# ------------------------------------------------------------

association_df[
    "Missing_All_Descriptors"
] = (
    association_df[
        descriptor_columns
    ]
    .isna()
    .all(axis=1)
)


# ------------------------------------------------------------
# Identify partially missing descriptor rows
# ------------------------------------------------------------

association_df[
    "Partially_Missing_Descriptors"
] = (
    association_df[
        descriptor_columns
    ]
    .isna()
    .any(axis=1)
    &
    ~association_df[
        descriptor_columns
    ]
    .isna()
    .all(axis=1)
)


# ============================================================
# Step 20.7
# Descriptor completeness statistics
# ============================================================

complete_descriptor_rows = (
    association_df[
        "Has_All_Descriptors"
    ]
    .sum()
)


all_missing_descriptor_rows = (
    association_df[
        "Missing_All_Descriptors"
    ]
    .sum()
)


partial_missing_descriptor_rows = (
    association_df[
        "Partially_Missing_Descriptors"
    ]
    .sum()
)


incomplete_descriptor_rows = (
    len(association_df)
    - complete_descriptor_rows
)


print("\nDescriptor Completeness")
print("-----------------------")

print(
    "Rows with all molecular descriptors:",
    complete_descriptor_rows
)

print(
    "Rows missing one or more descriptors:",
    incomplete_descriptor_rows
)

print(
    "Rows with all descriptors missing:",
    all_missing_descriptor_rows
)

print(
    "Rows with only some descriptors missing:",
    partial_missing_descriptor_rows
)


# ------------------------------------------------------------
# Percentage with complete descriptors
# ------------------------------------------------------------

if len(association_df) > 0:

    complete_percentage = (
        complete_descriptor_rows
        / len(association_df)
        * 100
    )

else:

    complete_percentage = 0


print(
    "Percentage of rows with complete descriptors:",
    f"{complete_percentage:.2f}%"
)


# ============================================================
# Step 20.8
# Create descriptor-complete dataset
# ============================================================

descriptor_complete_df = (
    association_df[
        association_df[
            "Has_All_Descriptors"
        ]
    ]
    .copy()
)


print("\nDescriptor-Complete Dataset")
print("---------------------------")

print(
    "Shape:",
    descriptor_complete_df.shape
)

print(
    "Number of drug–disease associations:",
    len(descriptor_complete_df)
)

print(
    "Unique DrugBank drugs:",
    descriptor_complete_df[
        "DrugBank_ID"
    ].nunique()
)

print(
    "Unique ChEMBL drugs:",
    descriptor_complete_df[
        "ChEMBL_ID"
    ].nunique()
)

print(
    "Unique diseases:",
    descriptor_complete_df[
        "diseaseId"
    ].nunique()
)


# ============================================================
# Step 20.9
# Compare dataset before and after descriptor filtering
# ============================================================

initial_associations = len(
    association_df
)

final_associations = len(
    descriptor_complete_df
)


initial_drugs = (
    association_df[
        "DrugBank_ID"
    ]
    .nunique()
)


final_drugs = (
    descriptor_complete_df[
        "DrugBank_ID"
    ]
    .nunique()
)


initial_diseases = (
    association_df[
        "diseaseId"
    ]
    .nunique()
)


final_diseases = (
    descriptor_complete_df[
        "diseaseId"
    ]
    .nunique()
)


print("\nBefore / After Descriptor Filtering")
print("-----------------------------------")

print(
    "Associations:",
    initial_associations,
    "→",
    final_associations
)

print(
    "Unique drugs:",
    initial_drugs,
    "→",
    final_drugs
)

print(
    "Unique diseases:",
    initial_diseases,
    "→",
    final_diseases
)


# ============================================================
# Step 20.10
# Clinical-stage distribution
# ============================================================

if "maxClinicalStage" in descriptor_complete_df.columns:

    print(
        "\nClinical Stage Distribution "
        "in Descriptor-Complete Dataset"
    )

    print(
        "-----------------------------------"
    )


    clinical_stage_counts = (
        descriptor_complete_df[
            "maxClinicalStage"
        ]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )


    print(
        clinical_stage_counts
    )


    # --------------------------------------------------------
    # Percentage distribution
    # --------------------------------------------------------

    clinical_stage_percentages = (
        descriptor_complete_df[
            "maxClinicalStage"
        ]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .sort_index()
        .mul(100)
        .round(2)
    )


    print(
        "\nClinical Stage Percentages:"
    )

    print(
        clinical_stage_percentages
    )


# ============================================================
# Step 20.11
# Preview descriptor-complete associations
# ============================================================

preview_columns = [
    "DrugBank_ID",
    "Drug_Name",
    "ChEMBL_ID",
    "diseaseId",
    "maxClinicalStage",
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Ring_Count"
]


available_preview_columns = [
    column
    for column in preview_columns
    if column in descriptor_complete_df.columns
]


print(
    "\nExample Descriptor-Complete "
    "Drug–Disease Associations:"
)


display(
    descriptor_complete_df[
        available_preview_columns
    ].head(10)
)


# ============================================================
# Step 20.12
# Final duplicate validation after descriptor filtering
# ============================================================

final_duplicate_count = (
    descriptor_complete_df
    .duplicated(
        subset=[
            "DrugBank_ID",
            "diseaseId"
        ]
    )
    .sum()
)


print("\nFinal Validation")
print("----------------")

print(
    "Duplicate DrugBank–disease pairs "
    "after descriptor filtering:",
    final_duplicate_count
)


if final_duplicate_count == 0:

    print(
        "The descriptor-complete dataset contains "
        "unique DrugBank_ID–diseaseId pairs."
    )

else:

    print(
        "Duplicated DrugBank_ID–diseaseId pairs "
        "remain and require further investigation."
    )


# ============================================================
# Step 20.13
# Save descriptor-complete dataset
# ============================================================

descriptor_complete_df.to_csv(
    OUTPUT_FILE,
    index=False
)


print(
    "\nDescriptor-complete dataset saved to:"
)

print(
    OUTPUT_FILE
)

Initial integrated dataset
--------------------------
Shape: (56044, 33)
Number of association rows: 56044
Unique DrugBank drugs: 5406
Unique ChEMBL drugs: 5406
Unique diseases: 3445

Drug–Disease Pair Validation
----------------------------
Duplicate DrugBank–disease rows: 0
Unique DrugBank–disease pairs: 56044
No duplicated DrugBank_ID–diseaseId pairs were found.

Molecular Descriptor Column Check
---------------------------------
All required molecular descriptor columns are present.

Molecular Descriptor Missing Values
-----------------------------------
MolWt: 6097 missing (10.88%)
LogP: 6097 missing (10.88%)
TPSA: 6097 missing (10.88%)
HBD: 6097 missing (10.88%)
HBA: 6097 missing (10.88%)
Rotatable_Bonds: 6097 missing (10.88%)
Ring_Count: 6097 missing (10.88%)

Descriptor Completeness
-----------------------
Rows with all molecular descriptors: 49947
Rows missing one or more descriptors: 6097
Rows with all descriptors missing: 6097
Rows with only some descriptors missing: 0
Perce

,DrugBank_ID,Drug_Name,ChEMBL_ID,diseaseId,maxClinicalStage,MolWt,LogP,TPSA,HBD,HBA,Rotatable_Bonds,Ring_Count
472,DB00014,Goserelin,CHEMBL1201247,MONDO_0007254,APPROVAL,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
473,DB00014,Goserelin,CHEMBL1201247,MONDO_0005070,APPROVAL,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
474,DB00014,Goserelin,CHEMBL1201247,MONDO_0005082,PHASE_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
475,DB00014,Goserelin,CHEMBL1201247,MONDO_0008315,PHASE_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
476,DB00014,Goserelin,CHEMBL1201247,EFO_0008008,PHASE_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
477,DB00014,Goserelin,CHEMBL1201247,MONDO_0007886,PHASE_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
478,DB00014,Goserelin,CHEMBL1201247,HP_0000131,PHASE_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
479,DB00014,Goserelin,CHEMBL1201247,MONDO_0005635,PHASE_2_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
480,DB00014,Goserelin,CHEMBL1201247,MONDO_0010888,PHASE_2_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0
481,DB00014,Goserelin,CHEMBL1201247,MONDO_0004992,PHASE_2_3,1269.433,-3.1057,495.89,17.0,16.0,31.0,6.0



Final Validation
----------------
Duplicate DrugBank–disease pairs after descriptor filtering: 0
The descriptor-complete dataset contains unique DrugBank_ID–diseaseId pairs.

Descriptor-complete dataset saved to:
/Users/sankhadeep/Desktop/DrugBank/drugbank_opentargets_descriptor_complete.csv
